# DINOv2 Token 群结构诊断

这个 notebook 不训练模型，也不依赖 decoder 重构图来判断群结构。它只回答一个更干净的问题：`RAE-DINOv2` 的 patch tokens 对旋转/翻转是否有可预测响应。

核心检查是 `E(gx)` 与 `P_g E(x)` 的 direct equivariance error，以及 token-to-token cosine correspondence。若这里显示 DINOv2 有稳定空间对应性，再进入 `E(gx) ≈ P_g E(x) C_g^T` 或 decoder adaptation 才自然。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "train_eqvae").exists() else CWD.parent
sys.path.insert(0, str(ROOT))

from baselines.dinov2_token_diagnostics import (
    NON_IDENTITY_TRANSFORMS,
    diagnostic_figure,
    diagnostic_table,
    load_dinov2_adapter,
    load_named_dataset,
    pick_dataset_images,
    procrustes_table,
    similarity_figure,
)

print(f"ROOT = {ROOT}")
print(f"CUDA = {torch.cuda.is_available()}, GPUs = {torch.cuda.device_count()}")

## 1. 配置

优先用清楚一点的自然图像数据，例如 Caltech101、Imagenette、Flowers102 或自定义 `image_folder`。CIFAR10 太小，token correspondence 很难肉眼判断。

In [ ]:
dataset_root = "/data/shared"
dataset_name = "caltech101"
dataset_split = "train"
dataset_path = ""     # dataset_name="image_folder" 时使用，例如 "/data/shared/imagenette2-320/train"
download_dataset = False

image_size = 256
count = 4
seed = 0
indices = None         # 例如 [0, 5, 9]；None 表示按 seed 随机

device_name = "cuda:0"
rae_repo_path = ROOT / "external/RAE"
rae_auto_clone = False
rae_auto_download = False

transforms = ("rot90", "rot180", "rot270", "flip_h")

In [ ]:
dataset = load_named_dataset(
    dataset_name,
    root=dataset_root,
    split=dataset_split,
    download=download_dataset,
    dataset_path=dataset_path,
)
x, selected_indices = pick_dataset_images(dataset, count=count, seed=seed, indices=indices, image_size=image_size)
adapter = load_dinov2_adapter(
    device=device_name,
    rae_repo_path=rae_repo_path,
    auto_clone=rae_auto_clone,
    auto_download=rae_auto_download,
)

print({"selected_indices": selected_indices, "x_shape": tuple(x.shape), "device": str(adapter.device)})

## 2. 直接等变与 token correspondence

`center="none"` 看原始 token；`center="sample"` 会去掉每张图的全局 token 均值，更强调空间 pattern。两者都值得看：如果 sample-centered 明显好，说明 DINOv2 有空间对应性，但被全局语义/位置偏置混在一起。

In [ ]:
raw = pd.DataFrame(diagnostic_table(adapter, x, transforms=transforms, center="none"))
raw.insert(0, "center", "none")
centered = pd.DataFrame(diagnostic_table(adapter, x, transforms=transforms, center="sample"))
centered.insert(0, "center", "sample")
display(pd.concat([raw, centered], ignore_index=True))

## 3. 轻量 channel 诊断：正交 Procrustes

这里不是训练模型，只是闭式求一个全局正交 channel rotation。如果它能大幅降低 centered error，说明 DINOv2 的响应里可能有干净的 channel-basis rotation；如果不能，结构可能更依赖 token/层/样本。

In [ ]:
display(pd.DataFrame(procrustes_table(adapter, x, transforms=transforms)))

## 4. 单样本可视化

面板依次显示：原图、变换图、`PCA(E(x))`、`PCA(E(gx))`、`PCA(P_gE(x))`、best-match displacement。若 DINOv2 有清晰空间对应，后两张 PCA 应相近，displacement 应集中在低值。

In [ ]:
sample_index = 0
g = "rot90"
diagnostic_figure(adapter, x, transform=g, sample_index=sample_index, center="sample")

## 5. 完整 similarity matrix

如果 token correspondence 接近理论位置，矩阵会接近对角结构；如果出现条纹、块状或远离对角的高相似度，说明 decoder 图里的条纹/错位更可能来自 token correspondence 本身不稳定。

In [ ]:
similarity_figure(adapter, x, transform=g, sample_index=sample_index, center="sample")